In [1]:
import os 
os.chdir("../")
%pwd

'd:\\Programming\\ML\\End-to-End\\End-to-End-TelcoChurn'

In [2]:
from dataclasses import dataclass 
from pathlib import Path 
from src import logging, CustomException



In [3]:
@dataclass(frozen=True)
class DataIngestionConfig:
  root_dir: Path
  source_url: str
  local_data_file: Path 
  unzip_dir: Path

In [4]:
from src.constants import * 
from src.utils import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self, 
                config_filepath:Path = Path(CONFIG_FILE_PATH),
                params_filepath:Path = Path(PARAMS_FILE_PATH),
                schema_filepath:Path = Path(SCHEMA_FILE_PATH)):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        return DataIngestionConfig(
            root_dir = Path(config.root_dir),
            source_url = str(config.source_url),
            local_data_file =  Path(config.local_data_file),
            unzip_dir = Path(config.unzip_dir)
        )

In [6]:
from urllib.request import urlretrieve
from src import logging, CustomException
import zipfile
import rarfile

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
        
    def download_file(self):
        try:
            if not os.path.exists(self.config.local_data_file):
                file_name, headers = urlretrieve(
                    url= self.config.source_url, 
                    filename=self.config.local_data_file
                )
                logging.info(f"{file_name} download!")
            else:
                logging.info(f"{self.config.local_data_file} already exist.")
        except Exception as e:
            raise CustomException(e) from e
            
    

    def extract_file(self):
        try:
            local_file = self.config.local_data_file
            unzip_dir = self.config.unzip_dir
            os.makedirs(unzip_dir, exist_ok=True)

            if zipfile.is_zipfile(local_file):
                with zipfile.ZipFile(local_file, "r") as f:
                    f.extractall(unzip_dir)
                logging.info("ZIP file extracted successfully.")

            elif rarfile.is_rarfile(local_file):
                with rarfile.RarFile(local_file, "r") as f:
                    f.extractall(unzip_dir)
                logging.info("RAR file extracted successfully.")

            else:
                logging.error(f"Unsupported or corrupt file: {local_file}")
                raise CustomException(f"Unsupported or corrupt file format: {local_file}")

        except Exception as e:
            raise CustomException(e) from e


In [8]:

config = ConfigurationManager()
data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)
data_ingestion.download_file()
data_ingestion.extract_file()

[2025-10-14 23:03:58,759] [INFO] [root:read_yaml:16] - reading the content of 'config\config.yaml'
[2025-10-14 23:03:58,761] [INFO] [root:read_yaml:16] - reading the content of 'params.yaml'
[2025-10-14 23:03:58,763] [INFO] [root:read_yaml:16] - reading the content of 'schema.yaml'
[2025-10-14 23:03:58,765] [INFO] [root:create_directories:39] - created directory at: artifacts


[2025-10-14 23:03:58,782] [INFO] [root:create_directories:39] - created directory at: artifacts/data_ingestion
[2025-10-14 23:04:01,824] [INFO] [root:download_file:13] - artifacts\data_ingestion\data.zip download! with the following info: 
Connection: close
Content-Length: 164404
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "357573f825b51cc8425e56f331cef41927db02df1283dfd6d348b7c4878d13ad"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 747E:2B6616:F31104:116E50F:68EEACAF
Accept-Ranges: bytes
Date: Tue, 14 Oct 2025 20:04:01 GMT
Via: 1.1 varnish
X-Served-By: cache-lin1730046-LIN
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1760472241.041828,VS0,VE174
Vary: Authorization,Accept-Encoding
Access-Control-Allow-Origin: *
Cross-Origin-Resource-Policy: cross-origin
X-Fastly-Request-ID: 5c6e31d0ca96645